In [3]:
# import required libraries
from kafka import KafkaConsumer, KafkaProducer
import avro.schema
import avro.io
import io
import hashlib, json

In [4]:
def serialize(schema, obj):
    bytes_writer = io.BytesIO()
    encoder = avro.io.BinaryEncoder(bytes_writer)
    writer = avro.io.DatumWriter(schema)
    writer.write(obj, encoder)
    return bytes_writer.getvalue()

In [5]:
def deserialize(schema, raw_bytes):
    bytes_reader = io.BytesIO(raw_bytes)
    decoder = avro.io.BinaryDecoder(bytes_reader)
    reader = avro.io.DatumReader(schema)
    return reader.read(decoder)

In [6]:
schema_file = 'transaction.avsc'
txschema = avro.schema.parse(open(schema_file).read())
schema_file = 'submit.avsc'
submitschema = avro.schema.parse(open(schema_file).read())
schema_file = 'result.avsc'
resultschema = avro.schema.parse(open(schema_file).read())

In [7]:
# Connect to kafka broker running in your local host (docker). Change this to your kafka broker if needed
kafka_broker = 'lab.aimet.tech:9092'
vid = "V505066"

In [8]:
token = "deac9b96c24a256a28a0f978bdb38d4c"

In [9]:
producer = KafkaProducer(bootstrap_servers=[kafka_broker])

In [10]:
txconsumer = KafkaConsumer(
    'transaction',
     bootstrap_servers=[kafka_broker],
     enable_auto_commit=True,
     value_deserializer=lambda x: deserialize(txschema, x))
resultconsumer = KafkaConsumer(
    'result',
     bootstrap_servers=[kafka_broker],
     enable_auto_commit=True,
     value_deserializer=lambda x: deserialize(resultschema, x))

In [11]:
def gen_signature(txid, payer, payee, amount, token):
    o = {'txid': txid, 'payer': payer, 'payee': payee, 'amount': amount, 'token': token}
    return hashlib.md5(json.dumps(o, sort_keys=True).encode('utf-8')).hexdigest()

In [24]:
for tx in txconsumer:
    txid, payer, payee, amount = tx.value.values()
    sign = gen_signature(txid, payer, payee, amount, token)
    submit_data = {
        "vid": vid,
        "txid": txid,
        "signature": sign,
    }
    payload = serialize(submitschema, submit_data)
    producer.send('submit', value=payload)
    for res in resultconsumer:
        timestamp, res_vid, res_txid, checksum, code, message = res.value.values()
        if res_txid == txid and res_vid == vid:
            if code == 200:
                print(res.value)
                with open('result_message.txt', 'a') as f:
                    f.write(str(res.value) + '\n')
            else:
                print("Code not 200", code)
            break


{'timestamp': 1774549013, 'vid': 'V505066', 'txid': 'TX00536', 'checksum': '58262418e87f7d17518df081fa8345cb', 'code': 200, 'message': 'Confirm'}
{'timestamp': 1774549014, 'vid': 'V505066', 'txid': 'TX03471', 'checksum': 'a6f3d16b6818aeb9044af39f43092950', 'code': 200, 'message': 'Confirm'}
{'timestamp': 1774549015, 'vid': 'V505066', 'txid': 'TX08554', 'checksum': '35d48e3242613fefddb664112466d809', 'code': 200, 'message': 'Confirm'}
{'timestamp': 1774549016, 'vid': 'V505066', 'txid': 'TX06021', 'checksum': 'bd44ed11eba901e965929cc1a2c3e540', 'code': 200, 'message': 'Confirm'}
{'timestamp': 1774549017, 'vid': 'V505066', 'txid': 'TX08221', 'checksum': '18297dc7b26008e011ed4bcc388b46ba', 'code': 200, 'message': 'Confirm'}
{'timestamp': 1774549018, 'vid': 'V505066', 'txid': 'TX02848', 'checksum': 'cc42dd7c7f8449333aff64a670b0fbc4', 'code': 200, 'message': 'Confirm'}
{'timestamp': 1774549019, 'vid': 'V505066', 'txid': 'TX04837', 'checksum': 'cdf4d887756f2ff72b2948b8151dcb4e', 'code': 200, 

KeyboardInterrupt: 